<a href="https://colab.research.google.com/github/antfolk/BMEN35_2026/blob/main/Session2/BMEN35_Ex2_01_Linear_regression_step_by_step.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Linear regression

In this notebook we will look at linear regression. We will start with a simple example and go through the calculations one step at a time. The first example uses a straight line. Later we will add a squared term and see how that lets us fit a curved relationship.

We will use NumPy arrays and Matplotlib, as in the earlier walkthroughs. The data are simulated, so we know how they were made and can compare our estimates with the values used to generate them.

## 1. Import libraries and make some data

Let us start by importing NumPy and Matplotlib. We make 500 observations. The input values are random numbers between 0 and 5. The outcome is made using an intercept of 2, a slope of -3 and some random noise.

The noise means that the observations will not fall exactly on a line. The model only sees the values in X and y; it does not get the intercept and slope we used to make them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Number of observations in this example.
N = 500

# Make input values between 0 and 5.
X = 5 * np.random.rand(N, 1)

# Make outcomes using an intercept, a slope, and random noise.
y = 2 - 3 * X + np.random.randn(N, 1)

print('Shape of X:', X.shape)
print('Shape of y:', y.shape)
print('First five input values:')
print(X[:5])
print('First five outcome values:')
print(y[:5])

## 2. Plot the observations

We want to examine the data before calculating anything. A scatter plot shows one point for each observation. The horizontal axis is X and the vertical axis is y.

In [ ]:
# Draw one point for each input and outcome pair.
plt.scatter(X, y)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Simulated observations')
plt.show()

The points have a downward trend. We can try to guess the intercept and slope by looking at the plot. The line used to create the data had intercept 2 and slope -3. The noise makes the exact pattern harder to see.

Next we will estimate the line from X and y. First we need to add a column of ones to X. This column will let the model include an intercept.

## 3. Add a column for the intercept

A linear model with one input can be written as:

$$
y = \theta_0 + \theta_1 x.
$$

Here, $\theta_0$ is the intercept and $\theta_1$ is the slope.

To use matrix multiplication, we put a 1 before each input value. Each row then has an intercept column and an X column. The corresponding parameter vector contains the intercept and slope in the same order.

In [ ]:
# Add a column of ones before the input column.
X_b = np.hstack((np.ones((N, 1)), X))

print('First five rows of X_b:')
print(X_b[:5])
print('Shape of X_b:', X_b.shape)

The first column is all ones. The second column contains the original input values. The first row, for example, represents the model $\theta_0 + \theta_1 x_1$.

Our data matrix has 500 rows and 2 columns. The estimated parameter vector will have 2 values: one for the intercept and one for the slope.

## 4. Calculate the parameters using the normal equation

For linear regression, we can calculate the least-squares parameters directly with the normal equation:

$$
\boldsymbol{\theta} = (\mathbf{X}_b^T\mathbf{X}_b)^{-1}\mathbf{X}_b^T\mathbf{y}.
$$

This equation can look complicated at first. We will calculate each part separately and print the shapes. The inside dimensions must match when we multiply matrices.

In [ ]:
# Transpose X_b: its shape changes from (N, 2) to (2, N).
X_b_transposed = X_b.T
print('Shape of X_b:', X_b.shape)
print('Shape of X_b_transposed:', X_b_transposed.shape)

# Multiply X_b transpose by X_b. The result has shape (2, 2).
X_b_transposed_X_b = np.dot(X_b_transposed, X_b)
print('X_b_transposed_X_b:')
print(X_b_transposed_X_b)
print('Shape:', X_b_transposed_X_b.shape)

Now we calculate the inverse of $X_b^T X_b$. We will keep this as a separate step so that the expression in the equation stays visible in the code.

In [ ]:
# Calculate the inverse of X_b transpose multiplied by X_b.
inverse_part = np.linalg.inv(X_b_transposed_X_b)
print('Inverse matrix:')
print(inverse_part)
print('Shape:', inverse_part.shape)

The next part is $X_b^T y$. Its shape is 2 by 1: one value for each model parameter. We can then multiply the inverse matrix by this result to get the intercept and slope.

In [ ]:
# Multiply X_b transpose by y.
X_b_transposed_y = np.dot(X_b_transposed, y)
print('X_b_transposed_y:')
print(X_b_transposed_y)
print('Shape:', X_b_transposed_y.shape)

## 5. Calculate and inspect the fitted parameters

Now we combine the parts from the normal equation. The first value is the estimated intercept and the second value is the estimated slope. Because the data include random noise, these estimates will be close to, but not necessarily exactly, 2 and -3.

In [ ]:
# Multiply the two parts to get the least-squares parameter estimates.
theta = np.dot(inverse_part, X_b_transposed_y)

print('Estimated intercept:')
print(theta[0])
print('Estimated slope:')
print(theta[1])
print('Both parameters [intercept, slope]:')
print(theta)

## 6. Make predictions with the fitted parameters

The model predicts $\hat{y}=\theta_0+\theta_1x$. We can use this equation for two input values, 0 and 5. Two points are enough to draw the fitted straight line over the range of our data.

In [ ]:
# These are the two input values where we want a prediction.
X_new = np.array([[0], [5]])

# Add a column of ones, just as we did for the training data.
X_new_b = np.hstack((np.ones((2, 1)), X_new))
print('New inputs with intercept column:')
print(X_new_b)

# Calculate the predicted y values using matrix multiplication.
y_hat = np.dot(X_new_b, theta)
print('Predictions at X=0 and X=5:')
print(y_hat)

Now we add the fitted line to the scatter plot. The points show the observations; the red line shows the values predicted by our model.

In [ ]:
# Plot the observations and the fitted regression line.
plt.scatter(X, y)
plt.plot(X_new, y_hat, 'r', linewidth=3)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Linear regression fitted to the observations')
plt.show()

## 7. Calculate residuals and mean squared error

A residual is the observed value minus the predicted value. We first use the fitted parameters to make a prediction for every original input value. Then we calculate the residual for each row.

Mean squared error (MSE) is the average of the squared residuals. It gives one summary of the size of the errors. Squaring makes all contributions positive and gives larger errors more influence.

In [ ]:
# Predict y for all observations used to fit the line.
y_hat_all = np.dot(X_b, theta)

# Calculate observed minus predicted for each observation.
residuals = y - y_hat_all

# Square each residual, then calculate the average.
squared_residuals = residuals ** 2
mse = np.mean(squared_residuals)

print('First five observed values:')
print(y[:5].flatten())  # X and y have shape (N, 1); flatten makes these five values easier to read as a 1D list.
print('First five predictions:')
print(y_hat_all[:5].flatten())  # This changes only the displayed shape, not the predicted values.
print('First five residuals:')
print(residuals[:5].flatten())  # Residuals also have shape (N, 1), so flatten keeps the printed output compact.
print('Mean squared error:')
print(mse)

## 8. Try polynomial regression

The previous example had a roughly straight-line pattern. What if the relationship is curved? We can create new simulated data using a squared term. We will first plot those data and then fit a model with an intercept, an $x^2$ term, and an $x$ term.

The model is still linear in its parameters, even though its curve is not a straight line in x.

In [ ]:
# Make a second simulated dataset with a curved relationship.
N = 500
X = 10 * np.random.rand(N, 1) - 5
y = 2 + 0.5 * X**2 + 3 * X + np.random.randn(N, 1)

print('Shape of X:', X.shape)
print('Shape of y:', y.shape)

# Plot the new observations before fitting a model.
plt.scatter(X, y)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Simulated observations with a curved pattern')
plt.show()

A straight line would not follow the curved pattern very well. We will add a column containing $x^2$. The columns will now be: ones for the intercept, squared input values, and the original input values.

In [ ]:
# Add columns for the intercept, X squared, and X.
X_b = np.hstack((np.ones((N, 1)), X**2, X))

print('First five rows [1, X squared, X]:')
print(X_b[:5])
print('Shape of the new design matrix:', X_b.shape)

We can use the same normal equation because the model is still a weighted sum of columns. There are now three parameters: intercept, coefficient for $x^2$, and coefficient for $x$.

In [ ]:
# Calculate the polynomial regression parameters step by step.
X_b_transposed = X_b.T
X_b_transposed_X_b = np.dot(X_b_transposed, X_b)
inverse_part = np.linalg.inv(X_b_transposed_X_b)
X_b_transposed_y = np.dot(X_b_transposed, y)
theta = np.dot(inverse_part, X_b_transposed_y)

print('Estimated parameters [intercept, coefficient for X squared, coefficient for X]:')
print(theta)

We can now draw the fitted curve. To make a smooth line, we create many X values from -5 to 5. For each value, we make the same three columns as before and calculate a prediction.

In [ ]:
# Make 500 evenly spaced input values for the fitted curve.
X_new = np.linspace(-5, 5, N).reshape(N, 1)

# Add the intercept and squared-value columns to the new inputs.
X_new_b = np.hstack((np.ones((N, 1)), X_new**2, X_new))

# Predict using the polynomial regression parameters.
y_hat = np.dot(X_new_b, theta)

# Plot the curved fit on top of the observations.
plt.scatter(X, y)
plt.plot(X_new, y_hat, 'r', linewidth=3)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Polynomial regression with an X squared term')
plt.show()

## 9. Fit a straight line using gradient descent

In the lecture we also discussed gradient descent. The normal equation gives us a direct solution for this small linear-regression problem. Gradient descent instead starts with parameter values and changes them step by step to reduce the error.

We will make another straight-line dataset. We will use a short sequence of code cells: first inspect the data, then make one update, and finally repeat the update for several epochs.

In [ ]:
# Make a third dataset for the gradient-descent example.
N = 500
X = 5 * np.random.rand(N, 1)
y = 1.8 + 3.4 * X + np.random.randn(N, 1)

# Look at this dataset before fitting a line.
plt.scatter(X, y)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Data for the gradient-descent example')
plt.show()

We will start the intercept and slope at zero. With these values, the first model predicts zero for every observation. Let us calculate and plot those initial predictions.

In [ ]:
# Start both model parameters at zero.
theta0 = 0.0  # Intercept
theta1 = 0.0  # Slope

# Calculate predictions from the starting model.
y_hat = theta0 + theta1 * X

# Calculate the mean squared error for these predictions.
n = len(X)
cost = np.sum((y - y_hat)**2) / n
print('Starting intercept:', theta0)
print('Starting slope:', theta1)
print('Starting mean squared error:', cost)

### One update, written out

The gradient tells us how the mean squared error changes with each parameter. For this model the two derivatives are:

$$
\frac{\partial MSE}{\partial \theta_0} = -\frac{2}{n}\sum_i(y_i-\hat{y}_i),\qquad
\frac{\partial MSE}{\partial \theta_1} = -\frac{2}{n}\sum_i x_i(y_i-\hat{y}_i).
$$

We calculate these two numbers and subtract the learning-rate-scaled derivatives from the parameters.

In [ ]:
# Calculate the error for the current predictions.
error = y - y_hat

# Calculate one derivative for the intercept and one for the slope.
D_theta0 = (-2 / n) * np.sum(error)
D_theta1 = (-2 / n) * np.sum(X * error)

# Use a small learning rate to update both parameters once.
eta = 0.01
theta0 = theta0 - eta * D_theta0
theta1 = theta1 - eta * D_theta1

print('Updated intercept:', theta0)
print('Updated slope:', theta1)

The parameters have changed a little. We can calculate the predictions and error again to see the effect of this one update.

In [ ]:
# Make new predictions using the updated parameters.
y_hat = theta0 + theta1 * X

# Calculate the new mean squared error.
new_cost = np.sum((y - y_hat)**2) / n
print('Mean squared error after one update:', new_cost)

## 10. Repeat the gradient-descent updates

One update is not enough to fit the line. We repeat the same operations for 20 epochs. In each epoch we calculate predictions using the current parameters, calculate the error and derivatives, and update both parameters. We save the parameter values and cost so we can inspect how they change.

In [ ]:
# Initialize the parameters again so training starts from zero.
theta0 = 0.0
theta1 = 0.0

# Set the learning rate and number of updates.
eta = 0.01
n_epochs = 20
n = len(X)

# Make arrays for saving the cost and parameters at every epoch.
cost = np.zeros(n_epochs)
thetas = np.zeros((n_epochs, 2))

# Repeat the gradient-descent update.
for i in range(n_epochs):
    # Predict with the current intercept and slope.
    y_hat = theta0 + theta1 * X

    # Calculate and save mean squared error.
    cost[i] = np.sum((y - y_hat)**2) / n

    # Calculate the two derivatives.
    D_theta0 = (-2 / n) * np.sum(y - y_hat)
    D_theta1 = (-2 / n) * np.sum(X * (y - y_hat))

    # Update the intercept and slope.
    theta0 = theta0 - eta * D_theta0
    theta1 = theta1 - eta * D_theta1

    # Save the updated parameters.
    thetas[i, :] = [theta0, theta1]

print('Final intercept:', theta0)
print('Final slope:', theta1)
print('Parameters at each epoch:')
print(thetas)

The parameters should move toward the values used to generate the data (intercept 1.8 and slope 3.4). The cost plot shows how the training error changes at each epoch.

In [ ]:
# Plot the mean squared error saved during training.
plt.plot(cost, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Mean squared error')
plt.title('Cost during gradient descent')
plt.grid(alpha=0.25)
plt.show()

## 11. Compare the fitted line with the data

We use the final gradient-descent parameters to draw a line. We also calculate parameters with the normal equation for this same dataset.

The normal equation calculates the least-squares answer directly for this linear model. Gradient descent starts at zero and moves toward a good answer in small steps. In this notebook we only use 20 epochs to make those steps easy to inspect. After 20 updates, gradient descent may not have reached the minimum yet, so its line can differ from the normal-equation line. More epochs usually bring it closer. A larger learning rate can also speed up learning, but if it is too large the updates may overshoot and the error can increase.

In [ ]:
# Make a line from the gradient-descent parameters.
X_new = np.linspace(0, 5, N).reshape(N, 1)
y_hat_gd = theta0 + theta1 * X_new

# Fit the same data with the normal equation for comparison.
X_b = np.hstack((np.ones((N, 1)), X))
theta_normal = np.dot(np.dot(np.linalg.inv(np.dot(X_b.T, X_b)), X_b.T), y)
y_hat_normal = np.dot(np.hstack((np.ones((N, 1)), X_new)), theta_normal)

print('Gradient-descent parameters [intercept, slope]:', theta0, theta1)
print('Normal-equation parameters:')
print(theta_normal)

In [ ]:
# Plot the observations and both fitted lines.
plt.scatter(X, y, label='observations')
plt.plot(X_new, y_hat_gd, 'r', linewidth=3, label='gradient descent')
plt.plot(X_new, y_hat_normal, 'g--', linewidth=2, label='normal equation')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Comparing two ways to fit a straight line')
plt.legend()
plt.show()

## 12. Questions about the examples

1. In the first example, what do the intercept and slope tell us about the relationship between X and y?
2. Why did we add a column of ones to the design matrix?
3. What does each row of the design matrix represent? What does each column represent?
4. How did adding an $x^2$ column change the shape of the fitted model?
5. In the gradient-descent example, what do the learning rate and number of epochs control?
6. Compare the gradient-descent line with the normal-equation line. Are they similar?

The normal equation gives a direct answer for this small linear-regression example. Gradient descent follows the same idea of reducing error, but takes repeated steps. Both examples show why it is useful to look at the data, print intermediate results, and plot the fitted model.